# Notebook 02 — Feature Engineering

Transforms the cleaned dataset into a model-ready feature matrix.

**Prerequisites:** Run `01_eda.ipynb` first (or `src/data/clean.py`) to ensure `data/samples/chicago_sample.csv` exists.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.features import engineer_features, get_feature_matrix, FEATURE_COLS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
os.makedirs('../outputs/figures', exist_ok=True)

## 1. Load cleaned sample

In [ ]:
df = pd.read_csv('../data/samples/chicago_sample.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print(f'Loaded {len(df):,} rows')

## 2. Apply feature engineering

In [ ]:
df_feat = engineer_features(df)
new_cols = ['Hour', 'DayOfWeek', 'Month', 'Year', 'IsWeekend', 'Season',
            'IsNight', 'PrimaryType_enc', 'LocationDesc_enc',
            'Season_enc', 'Domestic_enc']
print('New feature columns added:')
df_feat[new_cols].head()

## 3. Temporal feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, ['Hour', 'DayOfWeek', 'Month', 'Year', 'IsWeekend', 'IsNight']):
    df_feat[col].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Distribution of Temporal Features', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Arrest rate by temporal features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(
    axes,
    ['Hour', 'DayOfWeek', 'Month'],
    ['Hour of Day', 'Day of Week', 'Month']
):
    df_feat.groupby(col)['Arrest'].mean().plot(ax=ax, marker='o', color='crimson')
    ax.axhline(df_feat['Arrest'].mean(), ls='--', color='grey', label='Overall')
    ax.set_title(f'Arrest Rate by {title}')
    ax.set_ylabel('Arrest Rate')
    ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/arrest_rate_by_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Season analysis

In [ ]:
season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_counts = df_feat.groupby('Season').size().reindex(season_order)
season_arrest = df_feat.groupby('Season')['Arrest'].mean().reindex(season_order)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
season_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Crime Count by Season')
axes[0].set_xticklabels(season_order, rotation=0)

season_arrest.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Arrest Rate by Season')
axes[1].set_xticklabels(season_order, rotation=0)
axes[1].axhline(df_feat['Arrest'].mean(), ls='--', color='grey')

plt.tight_layout()
plt.savefig('../outputs/figures/season_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Build feature matrix and inspect

In [ ]:
X, y, feature_names = get_feature_matrix(df_feat)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}  |  Arrest rate: {y.mean():.1%}')
print(f'Features: {feature_names}')

In [ ]:
# Feature stats
pd.DataFrame(X, columns=feature_names).describe().T

## 7. Save feature-engineered dataset

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df_feat.to_csv('../data/processed/chicago_features.csv', index=False)
print(f'Saved feature-engineered data: {df_feat.shape}')